# Data preprocessing

### 1. Configuration


In [ ]:
# Import des biblioth?ques utiles
# Trouve le fichier de donn?es dans le bon dossier
# Cr?e le dossier de sortie
from pathlib import Path
import json

import joblib
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

DATASET_NAME = "healthcare-dataset-stroke-data.csv"
candidate_paths = [
    Path.cwd() / "data" / DATASET_NAME,
    Path.cwd().parent / "data" / DATASET_NAME,
    Path.cwd() / "data-science" / "data" / DATASET_NAME,
]
DATA_PATH = next((path.resolve() for path in candidate_paths if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"Could not find {DATASET_NAME}. Run this notebook from the repository root, "
        "the data-science directory, or the notebooks directory."
    )

OUTPUT_DIR = DATA_PATH.parent / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input: {DATA_PATH}")
print(f"Output directory: {OUTPUT_DIR}")


: 

### 2. Charger les données


In [ ]:
# Lit le CSV
# Affiche la forme du dataset
# V?rifie les valeurs manquantes
raw = pd.read_csv(DATA_PATH, na_values={"bmi": ["N/A"]})
print("1. DATASET SHAPE")
print(raw.shape)
print("\n2. FIRST ROWS")
display(raw.head())
print("\n3. MISSING VALUES")
display(raw.isna().sum().to_frame("missing_values"))


### 3. Nettoyer les données


In [ ]:
# Supprime la colonne inutile
# Retire la ligne "Other"
# Convertit les colonnes num?riques
# Remplace les valeurs BMI manquantes
data = raw.drop(columns="id").copy()

other_gender_count = int(data["gender"].eq("Other").sum())
data = data.loc[data["gender"].ne("Other")].copy()

data["age"] = pd.to_numeric(data["age"], errors="raise")
data["bmi"] = pd.to_numeric(data["bmi"], errors="coerce")

young_unknown_smoking = data["age"].lt(15) & data["smoking_status"].eq("Unknown")
data.loc[young_unknown_smoking, "smoking_status"] = "never smoked"

data["bmi"] = data["bmi"].clip(upper=60)
data["bmi_missing"] = data["bmi"].isna().astype("int8")

print(f"Removed gender='Other' rows: {other_gender_count}")
print(f"Recoded young unknown smoking rows: {int(young_unknown_smoking.sum())}")
print(f"BMI missing values: {int(data['bmi_missing'].sum())}")
print(f"Max BMI after capping: {data['bmi'].max():.1f}")


### 4. Encoder et séparer


In [ ]:
# Encode les variables en nombres
# S?pare les variables de la cible
# Divise les donn?es en train/test
binary_mappings = {
    "gender": {"Female": 0, "Male": 1},
    "ever_married": {"No": 0, "Yes": 1},
    "Residence_type": {"Rural": 0, "Urban": 1},
}

for column, mapping in binary_mappings.items():
    encoded = data[column].map(mapping)
    if encoded.isna().any():
        raise ValueError(f"Unexpected values in {column}: {sorted(data.loc[encoded.isna(), column].dropna().unique().tolist())}")
    data[column] = encoded.astype("int8")

data["stroke"] = pd.to_numeric(data["stroke"], errors="raise").astype("int8")
data = pd.get_dummies(data, columns=["work_type", "smoking_status"], drop_first=True, dtype="int8")

X = data.drop(columns="stroke")
y = data["stroke"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train stroke rate:", round(y_train.mean() * 100, 2), "%")
print("Test stroke rate:", round(y_test.mean() * 100, 2), "%")


### 5. Imputer et scaler


In [ ]:
# Remplace la BMI manquante avec la m?diane
# Standardise les variables num?riques
# V?rifie que la donn?e est bien transform?e
numeric_features = ["age", "avg_glucose_level", "bmi"]

bmi_imputer = SimpleImputer(strategy="median")
X_train.loc[:, "bmi"] = bmi_imputer.fit_transform(X_train[["bmi"]]).ravel()
X_test.loc[:, "bmi"] = bmi_imputer.transform(X_test[["bmi"]]).ravel()

scaler = StandardScaler()
X_train.loc[:, numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test.loc[:, numeric_features] = scaler.transform(X_test[numeric_features])

print(f"Imputed BMI median: {bmi_imputer.statistics_[0]:.2f}")
print("Missing BMI after imputation - train:", int(X_train["bmi"].isna().sum()))
print("Missing BMI after imputation - test:", int(X_test["bmi"].isna().sum()))


### 6. Enregistrer les résultats


In [ ]:
# Pr?pare les fichiers train/test
# Sauvegarde les mod?les et les donn?es
# Enregistre les infos utiles pour la suite
train_processed = X_train.copy()
train_processed["stroke"] = y_train
test_processed = X_test.copy()
test_processed["stroke"] = y_test

train_path = OUTPUT_DIR / "train_preprocessed.csv"
test_path = OUTPUT_DIR / "test_preprocessed.csv"
scaler_path = OUTPUT_DIR / "scaler.joblib"
imputer_path = OUTPUT_DIR / "bmi_imputer.joblib"
metadata_path = OUTPUT_DIR / "preprocessing_metadata.json"

train_processed.to_csv(train_path, index=False)
test_processed.to_csv(test_path, index=False)
joblib.dump(scaler, scaler_path)
joblib.dump(bmi_imputer, imputer_path)

metadata = {
    "target": "stroke",
    "binary_mappings": binary_mappings,
    "feature_columns": X.columns.tolist(),
    "numeric_features": numeric_features,
    "bmi_upper_clip": 60,
    "bmi_missing_indicator": "bmi_missing",
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved artifacts:")
for artifact in [train_path, test_path, scaler_path, imputer_path, metadata_path]:
    print(f"- {artifact} ({artifact.stat().st_size:,} bytes)")
